In [1]:
import ijson

# Carica solo il primo oggetto dal file JSON
with open("./research_products_data/research_products_complete.json", "r", encoding="utf-8") as f:
    first_product = next(ijson.items(f, "item"))

print("First research product keys and types:")
for key, value in first_product.items():
    print(f"{key}: {type(value).__name__}")

First research product keys and types:
authors: list
openAccessColor: str
publiclyFunded: bool
type: str
language: dict
countries: list
subjects: list
mainTitle: str
subTitle: NoneType
descriptions: list
publicationDate: str
publisher: NoneType
embargoEndDate: NoneType
sources: list
formats: list
contributors: NoneType
coverages: NoneType
bestAccessRight: dict
container: dict
documentationUrls: NoneType
codeRepositoryUrl: NoneType
programmingLanguage: NoneType
contactPeople: NoneType
contactGroups: NoneType
tools: NoneType
size: NoneType
version: NoneType
geoLocations: NoneType
id: str
originalIds: list
pids: list
dateOfCollection: NoneType
lastUpdateTimeStamp: NoneType
indicators: dict
projects: list
organizations: list
communities: list
collectedFrom: list
instances: list
isGreen: bool
isInDiamondJournal: bool
affiliated_organizations: list


In [2]:
import ijson
import json
from decimal import Decimal

def make_serializable(obj):
    if isinstance(obj, Decimal):
        return float(obj)
    elif isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [make_serializable(v) for v in obj]
    else:
        return obj

# Carica solo il primo oggetto dal file JSON
with open("./research_products_data/research_products_complete.json", "r", encoding="utf-8") as f:
    first_product = next(ijson.items(f, "item"))

serializable_product = make_serializable(first_product)

print("\nFirst research product (pretty print):")
print(json.dumps(serializable_product, ensure_ascii=False, indent=2))


First research product (pretty print):
{
  "authors": [
    {
      "fullName": "Alessandro Riga",
      "name": null,
      "surname": null,
      "rank": 1,
      "pid": {
        "id": {
          "scheme": "orcid",
          "value": "0000-0002-7240-0009"
        },
        "provenance": null
      }
    },
    {
      "fullName": "Irene Dori",
      "name": null,
      "surname": null,
      "rank": 2,
      "pid": {
        "id": {
          "scheme": "orcid",
          "value": "0000-0002-7579-4795"
        },
        "provenance": null
      }
    },
    {
      "fullName": "Stéphanie Vierin",
      "name": null,
      "surname": null,
      "rank": 3,
      "pid": null
    },
    {
      "fullName": "Giovanni Boschian",
      "name": null,
      "surname": null,
      "rank": 4,
      "pid": {
        "id": {
          "scheme": "orcid",
          "value": "0000-0003-4324-6353"
        },
        "provenance": null
      }
    },
    {
      "fullName": "Carlo Tozzi",
      

In [3]:
import ijson

null_counts = {}
total_count = 0

with open("./research_products_data/research_products_complete.json", "r", encoding="utf-8") as f:
    for product in ijson.items(f, "item"):
        total_count += 1
        for key in product:
            if key not in null_counts:
                null_counts[key] = 0
            if product[key] is None:
                null_counts[key] += 1

print("Percentuale di null value per ogni chiave:")
for key in null_counts:
    percent = (null_counts[key] / total_count) * 100 if total_count > 0 else 0
    print(f"{key}: {percent:.2f}%")

Percentuale di null value per ogni chiave:
authors: 0.32%
openAccessColor: 75.78%
publiclyFunded: 2.64%
type: 0.00%
language: 0.00%
countries: 10.71%
subjects: 33.93%
mainTitle: 0.00%
subTitle: 100.00%
descriptions: 30.62%
publicationDate: 0.03%
publisher: 27.12%
embargoEndDate: 95.26%
sources: 46.90%
formats: 72.31%
contributors: 90.34%
coverages: 100.00%
bestAccessRight: 2.69%
container: 50.21%
documentationUrls: 100.00%
codeRepositoryUrl: 100.00%
programmingLanguage: 100.00%
contactPeople: 99.99%
contactGroups: 100.00%
tools: 100.00%
size: 99.93%
version: 99.88%
geoLocations: 100.00%
id: 0.00%
originalIds: 0.00%
pids: 0.43%
dateOfCollection: 100.00%
lastUpdateTimeStamp: 100.00%
indicators: 0.03%
projects: 94.30%
organizations: 0.00%
communities: 62.73%
collectedFrom: 0.00%
instances: 0.00%
isGreen: 2.64%
isInDiamondJournal: 2.64%
affiliated_organizations: 0.00%


In [4]:
null_percentages = {key: float(value) / float(total_count) * 100 if total_count > 0 else 0.0
                    for key, value in null_counts.items()}

In [5]:
import ijson
import json

def filter_keys_by_null_percentage(input_path, output_path, null_percentages, threshold):
    keys_to_keep = [key for key, percent in null_percentages.items() if percent <= threshold]
    
    # Rimuovi le chiavi specificate
    keys_to_remove = [
        "language", "collectedFrom", "descriptions", "bestAccessRight", "countries", "subjects",
        "publisher", "isGreen", "isInDiamondJournal", "affiliated_organizations", "organizations", 
        "sources", "publiclyFunded", "originalIds", "instances", "pids"
    ]
    keys_to_keep = [key for key in keys_to_keep if key not in keys_to_remove]
    
    with open(input_path, "r", encoding="utf-8") as f_in, open(output_path, "w", encoding="utf-8") as f_out:
        f_out.write("[\n")  # Inizia array JSON
        first = True
        
        for product in ijson.items(f_in, "item", use_float=True):
            # Mantieni solo le entry con type = 'publication'
            if product.get('type') != 'publication':
                continue
                
            filtered_product = {key: product[key] for key in keys_to_keep if key in product}
            
            # Rimuovi "name", "surname", "provenance" da ogni author
            if "authors" in filtered_product and isinstance(filtered_product["authors"], list):
                for author in filtered_product["authors"]:
                    for key_to_remove in ["name", "surname", "provenance"]:
                        if key_to_remove in author:
                            del author[key_to_remove]
                    # Se esiste "pid" come dict, rimuovi "provenance" anche lì
                    if "pid" in author and isinstance(author["pid"], dict) and "provenance" in author["pid"]:
                        del author["pid"]["provenance"]

            # Estrai keyword, SDG, FOS, DDC, JEL e stampa altri scheme
            keywords = []
            sdg = []
            fos = []
            subjects = product.get("subjects", [])
            if isinstance(subjects, list):
                for subj in subjects:
                    subject_info = subj.get("subject")
                    if subject_info:
                        scheme = subject_info.get("scheme")
                        value = subject_info.get("value")
                        if scheme == "keyword":
                            if value:
                                keywords.append(value)
                        elif scheme == "SDG":
                            if value:
                                sdg.append(value)
                        elif scheme == "FOS":
                            if value:
                                fos.append(value)

            # Sempre presenti, anche se vuoti
            filtered_product["keywords"] = "; ".join(keywords) if keywords else ""
            filtered_product["SDG"] = "; ".join(sdg) if sdg else ""
            filtered_product["FOS"] = "; ".join(fos) if fos else ""
           
            if not first:
                f_out.write(",\n")  # Virgola tra elementi
            
            json.dump(filtered_product, f_out, ensure_ascii=False, indent=2)
            first = False
        
        f_out.write("\n]")  # Chiude array JSON

filter_keys_by_null_percentage(
    "./research_products_data/research_products_complete.json",
    "./research_products_data/research_products_filtered.json",
    null_percentages,
    threshold=50.0)

In [6]:
import ijson

null_counts = {}
total_count = 0

with open("./research_products_data/research_products_filtered.json", "r", encoding="utf-8") as f:
    for product in ijson.items(f, "item"):
        total_count += 1
        for key in product:
            if key not in null_counts:
                null_counts[key] = 0
            if product[key] is None:
                null_counts[key] += 1

print("Percentuale di null value per ogni chiave:")
for key in null_counts:
    percent = (null_counts[key] / total_count) * 100 if total_count > 0 else 0
    print(f"{key}: {percent:.2f}%")

Percentuale di null value per ogni chiave:
authors: 0.32%
type: 0.00%
mainTitle: 0.00%
publicationDate: 0.03%
id: 0.00%
indicators: 0.03%
keywords: 0.00%
SDG: 0.00%
FOS: 0.00%


In [7]:
import ijson
import json
from decimal import Decimal

def make_serializable(obj):
    if isinstance(obj, Decimal):
        return float(obj)
    elif isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [make_serializable(v) for v in obj]
    else:
        return obj

# Carica solo il primo oggetto dal file JSON
with open("./research_products_data/research_products_filtered.json", "r", encoding="utf-8") as f:
    first_product = next(ijson.items(f, "item"))

serializable_product = make_serializable(first_product)

print("\nFirst research product (pretty print):")
print(json.dumps(serializable_product, ensure_ascii=False, indent=2))


First research product (pretty print):
{
  "authors": [
    {
      "fullName": "Alessandro Riga",
      "rank": 1,
      "pid": {
        "id": {
          "scheme": "orcid",
          "value": "0000-0002-7240-0009"
        }
      }
    },
    {
      "fullName": "Irene Dori",
      "rank": 2,
      "pid": {
        "id": {
          "scheme": "orcid",
          "value": "0000-0002-7579-4795"
        }
      }
    },
    {
      "fullName": "Stéphanie Vierin",
      "rank": 3,
      "pid": null
    },
    {
      "fullName": "Giovanni Boschian",
      "rank": 4,
      "pid": {
        "id": {
          "scheme": "orcid",
          "value": "0000-0003-4324-6353"
        }
      }
    },
    {
      "fullName": "Carlo Tozzi",
      "rank": 5,
      "pid": null
    },
    {
      "fullName": "John C. Willman",
      "rank": 6,
      "pid": {
        "id": {
          "scheme": "orcid",
          "value": "0000-0001-7143-4533"
        }
      }
    },
    {
      "fullName": "Jacopo Mo